# UMAP and embeddings

Unreleased preview. Tested with Scanpy 1.11.5. Install the checkout with `pip install -e '.[scanpy]'` and use that environment as your Jupyter kernel.

Plot your Scanpy results with GenomeSpy. Scanpy computes the coordinates; GenomeSpy reads them.

This guide uses Scanpy’s packaged PBMC data, which already has UMAP coordinates. For your own data, run `sc.tl.umap(adata)` after your usual Scanpy preprocessing and neighbor calculation.


In [ ]:
import scanpy as sc
import genome_spy as gs
import genome_spy.integrations.scanpy as gsc

pbmc = sc.datasets.pbmc68k_reduced()

## Plot

Color cells by a cluster label or gene expression. Scroll over either plot to zoom both axes. Drag to pan. Hover over a cell for details.


In [ ]:
basic = gsc.pl.umap(
    pbmc,
    color=["louvain", "CD79A"],
    size=12,
    width=240,
    height=240,
    zoom=True,
    show=False,
)
basic

## Select cells

Drag a rectangle in the left plot. The same region appears in both panels, with a cell count below. Double-click in the left plot to clear it. Zoom is off in this example so dragging selects cells.


In [ ]:
brush = gs.selection_interval("cells", encodings=["x", "y"], on="mousedown", zoom=False)
linked = gsc.pl.umap(
    pbmc,
    color=["louvain", "CD79A"],
    selection=brush,
    size=12,
    width=240,
    height=240,
    zoom=False,
    show=False,
)

### Get the selected cells in Python

Run these cells in Jupyter. The web page can select and count cells; returning them to Python needs a running notebook.


In [ ]:
view = linked.widget(parameter_names=[brush.name], inline=True, controls=False)
view

Draw a selection, then run this cell to get its cell IDs and an AnnData subset. An empty selection returns no cells.


In [ ]:
mask = gsc.selection_mask(pbmc, view.parameter_values.get(brush.name))
selected_ids = pbmc.obs_names[mask]
selected_adata = pbmc[mask].copy()
selected_adata

## Customize

Set the plot size, point size and color map in the plotting call. Here, the color scale stops at the 99th percentile so a few high values do not dominate.


In [ ]:
custom = gsc.pl.umap(
    pbmc,
    color="CD79A",
    cmap="viridis",
    vmax="p99",
    size=12,
    frameon=False,
    title="CD79A expression",
    width=280,
    height=260,
    zoom=True,
    show=False,
)
custom

## Save a chart

Run this cell in Jupyter to save a self-contained HTML file. Hover and zoom work without a notebook server.


In [ ]:
custom.save("umap.html", inline=True)

## Set category colors

Choose colors by category name. The same names appear in the legend.


In [ ]:
phase_colors = gsc.pl.umap(
    pbmc,
    color="phase",
    palette={"G1": "#bcbd22", "S": "#17becf", "G2M": "#d62728"},
    size=12,
    width=280,
    height=260,
    zoom=True,
    show=False,
)
phase_colors

## Outline cells

Add outlines to separate nearby clusters. Hover and zoom work as before.


In [ ]:
outlines = gsc.pl.umap(
    pbmc,
    color="louvain",
    add_outline=True,
    size=12,
    width=280,
    height=260,
    zoom=True,
    show=False,
)
outlines

## Show the neighbor graph

Show edges from the neighbor graph already stored in the data. This call does not compute neighbors.


In [ ]:
edges = gsc.pl.umap(
    pbmc,
    color="phase",
    edges=True,
    edges_width=0.3,
    size=10,
    width=280,
    height=260,
    zoom=True,
    show=False,
)
edges

## Center a color scale

Use two colors for a stored score, with zero at the midpoint. The scores stay unchanged.


In [ ]:
centered = gsc.pl.umap(
    pbmc,
    color="S_score",
    cmap="redblue",
    vcenter=0,
    size=12,
    width=280,
    height=260,
    zoom=True,
    show=False,
)
centered

## Plot stored PCA coordinates

Use the same plotting options for PCA. Axis titles include the explained variance.


In [ ]:
pca = gsc.pl.pca(
    pbmc,
    color=["louvain", "CD79A"],
    annotate_var_explained=True,
    size=12,
    width=240,
    height=240,
    zoom=True,
    show=False,
)
pca

## Scatter cell annotations

Compare counts and detected genes using existing cell annotations. Color shows the mitochondrial percentage.


In [ ]:
cell_scatter = gsc.pl.scatter(
    pbmc,
    x="n_counts",
    y="n_genes",
    color="percent_mito",
    size=12,
    width=320,
    height=260,
    zoom=True,
    show=False,
)
cell_scatter

## Scatter gene annotations

Plot stored gene statistics instead of cells. Hover for the gene name and its values.


In [ ]:
gene_scatter = gsc.pl.scatter(
    pbmc,
    x="means",
    y="dispersions",
    color="dispersions_norm",
    use_raw=False,
    size=12,
    width=320,
    height=260,
    zoom=True,
    show=False,
)
gene_scatter

## Prepare embedding density

Compute density with Scanpy using the stored UMAP coordinates. This does not recalculate UMAP.


In [ ]:
sc.tl.embedding_density(pbmc, basis="umap", groupby="phase")

## Plot embedding density

Show S-phase cells by their density, with other cells in gray. Scroll to zoom and hover for cell details.


In [ ]:
density = gsc.pl.embedding_density(
    pbmc,
    basis="umap",
    groupby="phase",
    group="S",
    fg_dotsize=18,
    bg_dotsize=5,
    width=280,
    height=260,
    zoom=True,
    show=False,
)
density

:::{admonition} Data use and provenance
:class: note

This example uses 700 cells from Scanpy’s preprocessed [PBMC68k dataset](https://scanpy.readthedocs.io/en/stable/generated/scanpy.datasets.pbmc68k_reduced.html). It retains the supplied coordinates and annotations. Credit [10x Genomics](https://www.10xgenomics.com/datasets/fresh-68-k-pbm-cs-donor-a-1-standard-1-1-0) and Scanpy’s preprocessing contributors. Source data: [CC BY 4.0](https://creativecommons.org/licenses/by/4.0/).
Scanpy computes the density in the preparation cell above. The other examples read supplied coordinates and annotations.
:::
